# Ingest acquisition validation
A hand-built PDF goes through the same text extraction, identity check and promotion as `ingest acquire`.

In [ ]:
import json, shutil, sys, tempfile
from pathlib import Path
root = Path.cwd()
while not (root / "knowledge_palace").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from knowledge_palace.tests.test_ingest import BODY, DOI, minimal_pdf
from knowledge_palace.acquisition.ingest import acquire_source, companion_text, needs_ocr, pdf_page_texts
from knowledge_palace.acquisition.providers import LocalFileProvider
from knowledge_palace.acquisition.receipt import new_request
temp = tempfile.TemporaryDirectory()
base = Path(temp.name)
state, source = base / "state", base / "source"
state.mkdir(); source.mkdir()
pdf = base / "alpha.pdf"
pdf.write_bytes(minimal_pdf(["doi:%s %s" % (DOI, BODY), "Second page " + BODY]))

In [ ]:
pages = pdf_page_texts(pdf)
len(pages), [len(page) for page in pages], needs_ocr(pages)

In [ ]:
print(companion_text(pages)[:420])

In [ ]:
result = acquire_source(new_request(work_slug="alpha-2020-echo", ids={"doi": DOI}, local_path=pdf),
                        [LocalFileProvider()], state, source)
{key: result[key] for key in ("acquisition_status", "identity_status", "identity_evidence", "text_status", "claim_eligible", "next_action")}

In [ ]:
assert result["claim_eligible"]
result["promoted"], sorted(path.name for path in (source / "fulltext").iterdir())

In [ ]:
scan = base / "scan.pdf"
scan.write_bytes(minimal_pdf([""]))
scanned = acquire_source(new_request(work_slug="scan-2020-page", ids={"doi": DOI}, local_path=scan, user_confirmed_identity=True),
                         [LocalFileProvider()], state, source)
scanned["text_status"], scanned["claim_eligible"], scanned["next_action"]

In [ ]:
from knowledge_palace.acquisition.ingest import save_cards
from knowledge_palace.tests.test_ingest import MINI
vault = base / "vault"
shutil.copytree(MINI, vault)
good = vault / "papers" / "alpha-2020-echo.md"
bad = base / "alpha-copy.md"
bad.write_text(good.read_text(encoding="utf-8").replace("slug: alpha-2020-echo", "slug: alpha-copy"), encoding="utf-8")
report = save_cards([bad, good], vault, source)
assert [row["status"] for row in report["saved"]] == ["unchanged"]
report

In [ ]:
temp.cleanup()